In [57]:
import pandas as pd

In [58]:
df = pd.read_csv("spotify_millsongdata.csv")

In [59]:
df.head()

,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...


In [60]:
df.tail()

,artist,song,link,text
57645,Ziggy Marley,Good Old Days,/z/ziggy+marley/good+old+days_10198588.html,Irie days come on play \r\nLet the angels fly...
57646,Ziggy Marley,Hand To Mouth,/z/ziggy+marley/hand+to+mouth_20531167.html,Power to the workers \r\nMore power \r\nPowe...
57647,Zwan,Come With Me,/z/zwan/come+with+me_20148981.html,all you need \r\nis something i'll believe \...
57648,Zwan,Desire,/z/zwan/desire_20148986.html,northern star \r\nam i frightened \r\nwhere ...
57649,Zwan,Heartsong,/z/zwan/heartsong_20148991.html,come in \r\nmake yourself at home \r\ni'm a ...


In [61]:
df.shape

(57650, 4)

In [62]:
df.isnull().sum()

artist    0
song      0
link      0
text      0
dtype: int64

In [63]:
df = df.sample(5000).drop('link', axis=1).reset_index(drop=True)

In [64]:
df.head()

,artist,song,text
0,Kelly Family,Burning Fire,"My heart is true, but the road is long \r\nAn..."
1,Matt Monro,Over The Rainbow,Somewhere over the rainbow \r\nWay up high \...
2,Tom T. Hall,I'll Go Somewhere And Sing My Songs Again,Way out on the mountain near the sky hiding fr...
3,Rainbow,No Release,Through the smoke of dancers move \r\nDemonic...
4,Radiohead,I've Seen It All,"I've seen it all, I have seen the trees, \r\n..."


In [65]:
df['text'][0]

"My heart is true, but the road is long  \r\nAnd I'm a fool, 'cause I do you wrong  \r\nIt's hard today, to stay in line  \r\nBut I'll try it all, anything you want  \r\nYou got it all, you got it all today  \r\nYou got my soul  \r\nCHORUS:  \r\nBurning fire  \r\nBurning fire  \r\nBurning fire  \r\nI lost it all, so you can find it  \r\nIt's hard to breath in this world today  \r\nBut in mercy is the key of loving  \r\nYou got it all, you got it all today  \r\nYou got my soul  \r\nCHORUS  \r\nSo love it hides, behind your silence  \r\nAnd yesterday is gone today  \r\nSo I will love you now  \r\nLike nothing happened  \r\nCHORUS\r\n\r\n"

In [66]:
df = df.sample(5000)

Text cleaning / Text Preprocessing

In [67]:
df['text'] = df['text'].str.lower().replace(r'^\w\s', ' ').replace(r'\n', ' ', regex = True)

In [68]:
df.tail(5)

,artist,song,text
1628,Eminem,Above The Law,"[intro:] \r the poor stay poor, the rich get ..."
3073,Unseen,Stand Up And Fight,the 90's are almost over and hatred still runs...
1556,Violent Femmes,Forbidden,"come with us and play! \r see, we have breast..."
1106,Miley Cyrus,Do My Thang,every single night and every single day \r i'...
1737,New Order,Touched By The Hand Of God,i was standing by the ocean when i saw your fa...


In [69]:
import nltk
from nltk.stem.porter import PorterStemmer

In [70]:
stemmer = PorterStemmer()

In [71]:
def token(txt) :
    token = nltk.word_tokenize(txt)
    a = [stemmer.stem(w) for w in  token]
    return " ".join(a)

In [72]:
token("You are beautiful, beauty")

'you are beauti , beauti'

In [73]:
df['text'].apply(lambda x : token(x))

3372    ooh that sound good , whi do n't you turn that...
3478    not long ago while dancin ' our eye by chanc d...
2230    it happen i felt it happen i wa awak i wa n't ...
3633    goodnight , my angel , time to close your eye ...
2703    everi morn when the peopl are out and i 'm fre...
                              ...                        
1628    [ intro : ] the poor stay poor , the rich get ...
3073    the 90 's are almost over and hatr still run d...
1556    come with us and play ! see , we have breast a...
1106    everi singl night and everi singl day i'mma do...
1737    i wa stand by the ocean when i saw your face i...
Name: text, Length: 5000, dtype: object

In [74]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [75]:
tfid = TfidfVectorizer(analyzer='word', stop_words='english')

In [76]:
matrix = tfid.fit_transform(df['text'])

In [77]:
similar = cosine_similarity(matrix)

In [78]:
similar[0]

array([1.        , 0.017517  , 0.01786449, ..., 0.00280306, 0.01329637,
       0.00891162])

In [79]:
df[df['song'] == "I'll Do Anything" ].index[0]

4631

Recommender Function

In [83]:
def recommender(song_name) :
    idx = df[df['song'] == song_name].index[0]
    distance = sorted(list(enumerate(similar[idx])), reverse=True, key=lambda x: x[1])
    song = []
    for sId in distance[1:5] :
        song.append(df.iloc[sId[0]].song)
    return song

In [86]:
recommender("I'll Do Anything")

['Come On Home', 'Promise Me', 'Almost Home', 'Home']

In [87]:
import pickle

In [89]:
pickle.dump(similar, open("similarity", "wb"))

In [ ]:
pickle.dump(df, open())